# Extract question and answer descriptors

Regex-based approach

In [35]:
import pandas as pd 
import re
from collections import defaultdict

In [36]:
train_df = pd.read_csv("incidence_dataset.csv")
test_df = pd.read_csv("prediction_set.csv")

In [37]:
word_to_num_map = defaultdict(lambda x:x, {
    "one":1,
    "two":2,
    "three":3,
    "four":4,
    "five":5,
    "six":6,
    "seven":7,
    "eight":8,
    "nine":9,
    "ten":10,
    "eleven":11,
    "twelve":12
})


In [38]:
train_df.geography_name.unique()

<StringArray>
['Australia', 'New Zealand', 'United Kingdom', 'United States', 'Canada']
Length: 5, dtype: str

In [39]:
country_to_continent_ass = {
    'Australia': 'Asia-Pacific',
    'New Zealand': 'Asia-Pacific',
    'United Kingdom': 'Europe',
    'United States': 'North America',
    'Canada': 'North America',
}

In [40]:
train_df.geography_name.map(country_to_continent_ass).unique()

<StringArray>
['Asia-Pacific', 'Europe', 'North America']
Length: 3, dtype: str

In [41]:

# LLM-generated product features
dfcat = pd.read_json("product_features.json")

def preprocess(df):
    """
    Add features to incidence dataset for training
    """

    # extract timeframe in question
    period_info_df = df.question_text.str.lower().str.extract(r"in the past (?P<n_per>\d+|\w+) (?P<period>month|year|day)s?.*",expand=False)

    # convert timeframe to numeric
    timeframe_months = period_info_df.n_per.apply(lambda x: word_to_num_map[x] if x in word_to_num_map else x).astype(float)
    timeframe_months[period_info_df.period == "year"] *= 12
    df["timeframe_months"] = timeframe_months.fillna(12)

    # extract further question features
    df["have_you_quest"] = df.question_text.str.lower().str.contains(r"\bhave you\b")
    df["are_you_quest"] = df.question_text.str.lower().str.contains(r"\bare you\b")
    df["multi_choice_quest"] = df.question_text.str.lower().str.contains(r"\bwhich\b")
    df["multi_choice_quest"] |= df.question_text.str.lower().str.contains(r"\bplease select\b")
    df["do_you_own_quest"] = df.question_text.str.lower().str.contains(r"\bdo you own\b")

    # extract answer features
    df["n_qualifying"] = df.qualifying_options.apply(lambda x: len(x.split('|')))
    df["n_disqualifying"] = df.disqualifying_options.apply(lambda x: len(x.split('|')))
    df["qualifying_has_no"] = df.qualifying_options.str.lower().str.contains(r"\bno\b")
    df["qualifying_has_yes"] = df.qualifying_options.str.lower().str.contains(r"\byes\b")

    # group geographies into broader regions
    df["continent"] = df.geography_name.map(country_to_continent_ass)

    # merge product category features
    return df.merge(dfcat,left_on="category_name",right_on="product_category")
    

In [42]:
train_df_all = preprocess(train_df)
train_df_all.to_csv("train_augmented.csv")

In [43]:
test_df_all = preprocess(test_df)
test_df_all.to_csv("test_augmented.csv")